# Import libraries

In [1]:
import pandas as pd
import numpy as np
import ast
import random
from collections import Counter
from math import sqrt

# Load Datasets

In [2]:
df = pd.read_csv("E:/6thsem/HealthyBites/data/processed/healthybites_master_dataset_split.csv")

def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

df["core_ingredients"] = df["core_ingredients"].apply(safe_eval)
df["calories"] = pd.to_numeric(df["calories"], errors="coerce").fillna(0)

print("Total Recipes:", len(df))

Total Recipes: 200000


In [3]:
import sys
import os

# Adjust this path if needed
sys.path.append("..")

In [4]:
from core.tfidf_algorithm import calculate_idf
from core.tfidf_algorithm import create_sparse_vector
from core.tfidf_algorithm import calculate_cosine_similarity

# Reusing TF-IDF Logic

In [6]:
ALL_CORES = df["core_ingredients"].tolist()

IDF_WEIGHTS, VOCAB_MAP = calculate_idf(ALL_CORES)

RECIPE_VECTORS = [
    create_sparse_vector(r, IDF_WEIGHTS, VOCAB_MAP)
    for r in ALL_CORES
]

print("TF-IDF vectors created:", len(RECIPE_VECTORS))

TF-IDF vectors created: 200000


# Feature Generator

In [7]:
def compute_features(user_ing, recipe_ing, user_vec, recipe_vec, calories):
    
    user_set = set(user_ing)
    recipe_set = set(recipe_ing)
    
    overlap = len(user_set & recipe_set)
    missing = len(recipe_set - user_set)
    cosine = calculate_cosine_similarity(user_vec, recipe_vec)
    
    return [overlap, missing, cosine, calories]

In [8]:
X = []
y = []

NEGATIVE_SAMPLES = 5

for idx, row in df.iterrows():
    
    user_ing = row["core_ingredients"]
    user_vec = RECIPE_VECTORS[idx]
    
    # Positive sample
    features = compute_features(
        user_ing,
        row["core_ingredients"],
        user_vec,
        RECIPE_VECTORS[idx],
        row["calories"]
    )
    
    X.append(features)
    y.append(1)
    
    # Negative samples
    for _ in range(NEGATIVE_SAMPLES):
        
        rand_idx = random.randint(0, len(df)-1)
        
        if rand_idx == idx:
            continue
        
        rand_row = df.iloc[rand_idx]
        
        features = compute_features(
            user_ing,
            rand_row["core_ingredients"],
            user_vec,
            RECIPE_VECTORS[rand_idx],
            rand_row["calories"]
        )
        
        X.append(features)
        y.append(0)

X = np.array(X)
y = np.array(y)

print("Training samples:", len(X))

Training samples: 1199995


Gini

In [9]:
def gini(y):
    classes = np.unique(y)
    impurity = 1
    for cls in classes:
        p = np.sum(y == cls) / len(y)
        impurity -= p**2
    return impurity

Best split

In [10]:
def split(X, y, feature, threshold):
    left = X[:, feature] <= threshold
    right = X[:, feature] > threshold
    return X[left], X[right], y[left], y[right]

def best_split(X, y, max_features):
    best_gini = float("inf")
    best_feature = None
    best_threshold = None
    
    features = random.sample(range(X.shape[1]), max_features)
    
    for f in features:
        thresholds = np.unique(X[:, f])
        
        for t in thresholds:
            X_l, X_r, y_l, y_r = split(X, y, f, t)
            
            if len(y_l) == 0 or len(y_r) == 0:
                continue
            
            g = (len(y_l)/len(y))*gini(y_l) + (len(y_r)/len(y))*gini(y_r)
            
            if g < best_gini:
                best_gini = g
                best_feature = f
                best_threshold = t
                
    return best_feature, best_threshold

# Decision Tree

In [11]:
class DecisionTree:
    
    def __init__(self, max_depth=6, min_samples=10, max_features=2):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.max_features = max_features
    
    def fit(self, X, y, depth=0):
        
        if depth >= self.max_depth or len(y) < self.min_samples or gini(y) == 0:
            self.label = Counter(y).most_common(1)[0][0]
            self.feature = None
            return
        
        feature, threshold = best_split(X, y, self.max_features)
        
        if feature is None:
            self.label = Counter(y).most_common(1)[0][0]
            self.feature = None
            return
        
        self.feature = feature
        self.threshold = threshold
        
        X_l, X_r, y_l, y_r = split(X, y, feature, threshold)
        
        self.left = DecisionTree(self.max_depth, self.min_samples, self.max_features)
        self.right = DecisionTree(self.max_depth, self.min_samples, self.max_features)
        
        self.left.fit(X_l, y_l, depth+1)
        self.right.fit(X_r, y_r, depth+1)
    
    def predict_one(self, x):
        if self.feature is None:
            return self.label
        if x[self.feature] <= self.threshold:
            return self.left.predict_one(x)
        else:
            return self.right.predict_one(x)
    
    def predict(self, X):
        return np.array([self.predict_one(x) for x in X])

# Random Forest

In [12]:
class RandomForest:
    
    def __init__(self, n_trees=20, max_depth=6):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []
    
    def fit(self, X, y):
        n_samples = len(X)
        
        for _ in range(self.n_trees):
            indices = np.random.choice(n_samples, n_samples, replace=True)
            X_sample = X[indices]
            y_sample = y[indices]
            
            tree = DecisionTree(max_depth=self.max_depth)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)
    
    def predict(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees])
        return np.round(np.mean(predictions, axis=0)).astype(int)
    
    def predict_proba(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees])
        return np.mean(predictions, axis=0)

# Train/Test split

In [13]:
split_idx = int(0.8 * len(X))

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

In [15]:
# Downsample to manageable size
MAX_SAMPLES = 20000   # adjust if needed

if len(X_train) > MAX_SAMPLES:
    indices = np.random.choice(len(X_train), MAX_SAMPLES, replace=False)
    X_train_small = X_train[indices]
    y_train_small = y_train[indices]
else:
    X_train_small = X_train
    y_train_small = y_train

print("Training on samples:", len(X_train_small))

Training on samples: 20000


# Model Train

In [16]:
rf = RandomForest(
    n_trees=10,      # reduce trees
    max_depth=5      # reduce depth
)

rf.fit(X_train_small, y_train_small)

# Accuracy

In [17]:
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)

accuracy = np.sum(y_pred == y_test) / len(y_test)
print("Accuracy:", accuracy)

Accuracy: 0.9998208325868024


Confusion Matrix

In [18]:
TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == 0) & (y_pred == 0))
FP = np.sum((y_test == 0) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == 0))

print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)

TP: 40000
TN: 199956
FP: 43
FN: 0


In [19]:
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 0.9989261543840372
Recall: 1.0
F1 Score: 0.9994627887510464


In [20]:
print("Sample confidence scores:")
print(y_prob[:10])

Sample confidence scores:
[1. 0. 0. 0. 0. 0. 1. 0. 0. 0.]


In [21]:
print("Train Accuracy:", np.mean(rf.predict(X_train_small) == y_train_small))
print("Test Accuracy:", np.mean(rf.predict(X_test) == y_test))

Train Accuracy: 0.99995
Test Accuracy: 0.9998208325868024
